In [6]:
import sqlite3
import pandas as pd

# Display numbers in readable format instead of scientific notation
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# bikin koneksi ke database (file baru, otomatis dibuat kalau belum ada)
conn = sqlite3.connect("nike_finance.db")

# load tiap CSV jadi satu tabel
income_stmt = pd.read_csv("nike_income_statement_raw.csv", index_col=0)
balance_sheet = pd.read_csv("nike_balance_sheet_raw.csv", index_col=0)
cash_flow = pd.read_csv("nike_cash_flow_raw.csv", index_col=0)
peer_metrics = pd.read_csv("peer_metrics_raw.csv", index_col=0)

income_stmt.to_sql("income_statement", conn, if_exists="replace")
balance_sheet.to_sql("balance_sheet", conn, if_exists="replace")
cash_flow.to_sql("cash_flow", conn, if_exists="replace")
peer_metrics.to_sql("peer_metrics", conn, if_exists="replace")

print("✓ Semua tabel berhasil dimuat ke database")

✓ Semua tabel berhasil dimuat ke database


In [7]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

                    name
0  income_statement_long
1       income_statement
2          balance_sheet
3              cash_flow
4           peer_metrics


In [8]:
# Pull Nike's Total Revenue across all years, ordered by date descending
query = """
SELECT "index" AS metric, "2023-05-31", "2024-05-31", "2025-05-31", "2026-05-31"
FROM income_statement
WHERE "index" = 'Total Revenue';
"""

revenue_from_sql = pd.read_sql(query, conn)
print(revenue_from_sql)

          metric        2023-05-31        2024-05-31        2025-05-31  \
0  Total Revenue 51,217,000,000.00 51,362,000,000.00 46,309,000,000.00   

         2026-05-31  
0 46,398,000,000.00  


In [9]:
# Reshape income_statement from wide (years as columns) to long format (years as rows)
# This makes filtering and joining far more natural in SQL
income_stmt_long = income_stmt.reset_index().melt(
    id_vars="index",
    var_name="fiscal_year",
    value_name="value"
)
income_stmt_long.columns = ["metric", "fiscal_year", "value"]

income_stmt_long.to_sql("income_statement_long", conn, if_exists="replace", index=False)

print(income_stmt_long.head(10))

                                              metric fiscal_year  \
0                        Tax Effect Of Unusual Items  2026-05-31   
1                                 Tax Rate For Calcs  2026-05-31   
2                                  Normalized EBITDA  2026-05-31   
3  Net Income From Continuing Operation Net Minor...  2026-05-31   
4                            Reconciled Depreciation  2026-05-31   
5                         Reconciled Cost Of Revenue  2026-05-31   
6                                             EBITDA  2026-05-31   
7                                               EBIT  2026-05-31   
8                                Net Interest Income  2026-05-31   
9                                  Normalized Income  2026-05-31   

              value  
0              0.00  
1              0.20  
2  4,594,000,000.00  
3  3,108,000,000.00  
4    797,000,000.00  
5 26,487,000,000.00  
6  4,594,000,000.00  
7  3,797,000,000.00  
8     50,000,000.00  
9  3,108,000,000.00  


In [10]:
print(f"Total rows: {len(income_stmt_long)}")

Total rows: 190


In [11]:
# Self-join: compute Gross Margin directly in SQL by joining
# the "Total Revenue" rows to the "Gross Profit" rows on fiscal_year
query = """
SELECT
    rev.fiscal_year,
    rev.value AS total_revenue,
    gp.value AS gross_profit,
    ROUND(gp.value * 100.0 / rev.value, 2) AS gross_margin_pct
FROM income_statement_long AS rev
JOIN income_statement_long AS gp
    ON rev.fiscal_year = gp.fiscal_year
WHERE rev.metric = 'Total Revenue'
  AND gp.metric = 'Gross Profit'
ORDER BY rev.fiscal_year DESC;
"""

margin_from_sql = pd.read_sql(query, conn)
print(margin_from_sql)

  fiscal_year     total_revenue      gross_profit  gross_margin_pct
0  2026-05-31 46,398,000,000.00 19,911,000,000.00             42.91
1  2025-05-31 46,309,000,000.00 19,790,000,000.00             42.73
2  2024-05-31 51,362,000,000.00 22,887,000,000.00             44.56
3  2023-05-31 51,217,000,000.00 22,292,000,000.00             43.52
4  2022-05-31               NaN               NaN               NaN


In [14]:
query = """
SELECT
    AVG(ROUND(gp.value * 100.0 / rev.value, 2)) AS avg_gross_margin_pct
FROM income_statement_long AS rev
JOIN income_statement_long AS gp
    ON rev.fiscal_year = gp.fiscal_year
WHERE rev.metric = 'Total Revenue'
  AND gp.metric = 'Gross Profit';
"""
avg_margin = pd.read_sql(query, conn)
print(avg_margin)

   avg_gross_margin_pct
0                 43.43


In [15]:
query = """
SELECT
    metric,
    AVG(value) AS avg_value
FROM income_statement_long
WHERE fiscal_year != '2022-05-31'
GROUP BY metric
ORDER BY avg_value DESC
LIMIT 10;
"""
avg_by_metric = pd.read_sql(query, conn)
print(avg_by_metric)

                               metric         avg_value
0                       Total Revenue 48,821,500,000.00
1                   Operating Revenue 48,821,500,000.00
2                      Total Expenses 43,890,250,000.00
3          Reconciled Cost Of Revenue 27,601,500,000.00
4                     Cost Of Revenue 27,601,500,000.00
5                        Gross Profit 21,220,000,000.00
6  Selling General And Administration 16,288,750,000.00
7                   Operating Expense 16,288,750,000.00
8            Other Operating Expenses 12,317,000,000.00
9                        Other Gand A 11,841,750,000.00


In [16]:
balance_sheet_long = balance_sheet.reset_index().melt(
    id_vars="index", var_name="fiscal_year", value_name="value"
)
balance_sheet_long.columns = ["metric", "fiscal_year", "value"]
balance_sheet_long.to_sql("balance_sheet_long", conn, if_exists="replace", index=False)
print(balance_sheet_long.head(10))

                      metric fiscal_year             value
0     Ordinary Shares Number  2026-05-31  1,483,498,703.00
1               Share Issued  2026-05-31  1,483,498,703.00
2                   Net Debt  2026-05-31    379,000,000.00
3                 Total Debt  2026-05-31 11,033,000,000.00
4        Tangible Book Value  2026-05-31 14,366,000,000.00
5           Invested Capital  2026-05-31 22,807,000,000.00
6            Working Capital  2026-05-31 12,056,000,000.00
7        Net Tangible Assets  2026-05-31 14,366,000,000.00
8  Capital Lease Obligations  2026-05-31  3,091,000,000.00
9        Common Stock Equity  2026-05-31 14,865,000,000.00


In [17]:
query = """
SELECT
    ni.fiscal_year,
    ni.value AS net_income,
    ta.value AS total_assets,
    ROUND(ni.value * 100.0 / ta.value, 2) AS roa_pct
FROM income_statement_long AS ni
JOIN balance_sheet_long AS ta
    ON ni.fiscal_year = ta.fiscal_year
WHERE ni.metric = 'Net Income'
  AND ta.metric = 'Total Assets'
ORDER BY ni.fiscal_year DESC;
"""
roa_from_sql = pd.read_sql(query, conn)
print(roa_from_sql)

  fiscal_year       net_income      total_assets  roa_pct
0  2026-05-31 3,108,000,000.00 38,410,000,000.00     8.09
1  2025-05-31 3,219,000,000.00 36,579,000,000.00     8.80
2  2024-05-31 5,700,000,000.00 38,110,000,000.00    14.96
3  2023-05-31 5,070,000,000.00 37,531,000,000.00    13.51
4  2022-05-31              NaN               NaN      NaN


In [18]:
query = """
SELECT
    fiscal_year,
    value AS total_revenue,
    LAG(value) OVER (ORDER BY fiscal_year) AS prior_year_revenue,
    ROUND(
        (value - LAG(value) OVER (ORDER BY fiscal_year)) * 100.0
        / LAG(value) OVER (ORDER BY fiscal_year), 2
    ) AS yoy_growth_pct
FROM income_statement_long
WHERE metric = 'Total Revenue' AND value IS NOT NULL
ORDER BY fiscal_year;
"""
yoy_growth = pd.read_sql(query, conn)
print(yoy_growth)

  fiscal_year     total_revenue  prior_year_revenue  yoy_growth_pct
0  2023-05-31 51,217,000,000.00                 NaN             NaN
1  2024-05-31 51,362,000,000.00   51,217,000,000.00            0.28
2  2025-05-31 46,309,000,000.00   51,362,000,000.00           -9.84
3  2026-05-31 46,398,000,000.00   46,309,000,000.00            0.19
